#  Linear Baseline and Multimodal Deep Learning

This notebook fits two models on the cleaned, deduplicated USD listings: a Ridge-regularized linear regression baseline using structured features and train-only TF-IDF text, and a multimodal neural regressor combining structured features, description embeddings, and house photos. The implementation is in [`src/train_models.py`](../src/train_models.py).

## Architecture and modeling choices

- **Target:** `log1p(price_usd)` reduces the effect of the strongly right-skewed asking prices. Scores are transformed back to USD for interpretation.
- **Split:** a seeded, price-decile-stratified train/validation/test split. Numeric scaling, category levels, text vocabulary, and TF-IDF statistics are learned from training rows only. Duplicate URLs were already consolidated in the cleaning notebook.
- **Linear baseline:** Ridge-regularized linear regression over standardized bedrooms/bathrooms/areas, one-hot location/property-type/amenity features, and a compact TF-IDF description representation. L2 regularization stabilizes correlated and sparse features.
- **Multimodal model:** a frozen ImageNet-pretrained ResNet18 extracts a 512-dimensional CNN feature from each listing photo. A trainable word-embedding branch mean-pools up to 192 description tokens. A structured MLP, text projection, and image projection are concatenated and passed to a fusion MLP. The model starts from the fitted linear baseline and learns a multimodal residual, so early stopping can retain the baseline if extra modalities do not help validation performance.
- **Training:** Smooth L1 loss on log-price, AdamW, gradient clipping, validation-based early stopping, and a fixed seed. The image encoder is frozen to reduce overfitting on this small dataset; only the embedding and fusion layers are trained.
- **Leakage control:** currency-price expressions in text are masked; the target and listing identifiers are not model inputs. The held-out test set is used only for final evaluation.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

HERE = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in [HERE, *HERE.parents] if (path / 'reports' / 'eda' / 'cleaned_listings.csv').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run notebooks/01_clean_explore.ipynb first to create reports/eda/cleaned_listings.csv.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.train_models import run_training

results = run_training(PROJECT_ROOT, max_epochs=60, seed=42)
display(results['metrics'])

Rows: 2,618; train/validation/test: 1,781/315/522
Listings with a local image: 2,607/2,618
Multimodal best epoch: 26
                                      model      MAE_USD     RMSE_USD  R2_USD  MAE_log
Multimodal ResNet18 + text embeddings + MLP 104,521.0078 217,604.0938  0.6239   0.3454
         Linear Ridge (structured + TF-IDF) 109,676.0703 234,134.8594  0.5646   0.3581
Reports: C:\Users\Kupa\Desktop\Zimbabwe-House-Price-Prediction\reports\modeling
Saved models: C:\Users\Kupa\Desktop\Zimbabwe-House-Price-Prediction\models\linear_baseline.npz, C:\Users\Kupa\Desktop\Zimbabwe-House-Price-Prediction\models\multimodal_fusion.pt


,model,MAE_USD,RMSE_USD,R2_USD,MAE_log
0,Multimodal ResNet18 + text embeddings + MLP,104521.007812,217604.093750,0.623949,0.345429
1,Linear Ridge (structured + TF-IDF),109676.070312,234134.859375,0.564643,0.358112


## Held-out evaluation

MAE and RMSE are in USD; R² is computed in USD space. Because of the log target, both models optimize a scale-aware loss but are judged in the original price currency. Compare the multimodal model with the linear baseline rather than assuming the neural model must win. The dataset is geographically concentrated and listing photos/descriptions may be incomplete or noisy, so report the split and limitations with any conclusions.

In [2]:
display(results['metrics'].sort_values('MAE_USD'))
display(results['predictions'].head(10))
display(results['history'].tail())
print('\nSaved artifacts:')
for name, path in results['paths'].items():
    print(f'{name}: {path}')

,model,MAE_USD,RMSE_USD,R2_USD,MAE_log
0,Multimodal ResNet18 + text embeddings + MLP,104521.007812,217604.093750,0.623949,0.345429
1,Linear Ridge (structured + TF-IDF),109676.070312,234134.859375,0.564643,0.358112


,listing_id,actual_price_usd,linear_baseline_price_usd,multimodal_price_usd
0,257770,11000.0,23467.683594,23299.976562
1,43251732,42000.0,44361.363281,45454.808594
2,256067,25000.0,84034.851562,33750.906250
3,258686,35000.0,98064.703125,136878.937500
4,241684,45000.0,52821.167969,39906.566406
5,259610,43000.0,93989.742188,70221.218750
6,258931,43000.0,97977.203125,64722.886719
7,254422,25000.0,75336.585938,36767.640625
8,254224,26500.0,43131.964844,34470.144531
9,248924,45000.0,40614.656250,42275.355469


,epoch,train_loss,validation_loss
31,32,0.009811,0.198733
32,33,0.009005,0.201554
33,34,0.010096,0.198504
34,35,0.008985,0.198122
35,36,0.007584,0.199869



Saved artifacts:
metrics: C:\Users\Kupa\Desktop\Zimbabwe-House-Price-Prediction\reports\modeling\model_comparison.csv
predictions: C:\Users\Kupa\Desktop\Zimbabwe-House-Price-Prediction\reports\modeling\test_predictions.csv
training_history: C:\Users\Kupa\Desktop\Zimbabwe-House-Price-Prediction\reports\modeling\training_history.csv
prediction_plot: C:\Users\Kupa\Desktop\Zimbabwe-House-Price-Prediction\reports\modeling\actual_vs_predicted.png
history_plot: C:\Users\Kupa\Desktop\Zimbabwe-House-Price-Prediction\reports\modeling\training_history.png
linear_model: C:\Users\Kupa\Desktop\Zimbabwe-House-Price-Prediction\models\linear_baseline.npz
multimodal_model: C:\Users\Kupa\Desktop\Zimbabwe-House-Price-Prediction\models\multimodal_fusion.pt
metadata: C:\Users\Kupa\Desktop\Zimbabwe-House-Price-Prediction\models\model_metadata.json
image_cache: C:\Users\Kupa\Desktop\Zimbabwe-House-Price-Prediction\models\cache\resnet18_36d0908ffa7d7bbd.npz


## Residual diagnostics and price-band errors

Residuals are defined as **actual minus predicted**, so positive values indicate underprediction. The plots check residuals in USD against fitted values and show their distribution on the log scale used for training. The price-band table helps reveal whether average performance hides larger errors for expensive listings.

In [ ]:
from IPython.display import Image

display(results['residual_metrics'].round(3))
display(results['errors_by_price_band'].round(2))
display(Image(filename=str(results['paths']['residual_plot'])))

## Assessment and limitations

On the 522-listing hold-out set, the multimodal model improves over Ridge: MAE is about **$104,521 vs. $109,676**, RMSE **$217,604 vs. $234,135**, and R² **0.624 vs. 0.565**. That is a modest single-split gain, not evidence that the extra modalities will generalize better everywhere.

The residual summaries show a mean residual near **+$22,000** for both models (slight overall underprediction), while median residuals are much closer to zero. Price-band errors show the strongest limitation: MAE rises from about **$30,000–$32,000** in the lowest-price band to **$298,000–$330,000** in the highest-price band. The multimodal model is not better in every band; its MAE is higher in the $155,000–$250,000 band. Residuals are generally negative at the low end and positive at the high end, consistent with overpredicting cheaper homes and underpredicting expensive homes. The large gap between MAE and RMSE also indicates a few large misses; price errors are heteroscedastic and are not well summarized by one average.

These are asking-price estimates, not transaction-price estimates. The data are geographically concentrated (especially Harare), contain missing or noisy listing details, and some records lack usable photos/text. The vision features use a frozen ImageNet ResNet18 rather than a property-specific encoder, and the text embeddings are learned from this relatively small training set rather than a pretrained transformer. Finally, this is one seeded, price-stratified hold-out split, not cross-validation; repeated or time-based splits and more geographically diverse, verified sale prices are needed before claiming robust generalization.

## Model-specific caveats

Of the 2,618 eligible listings, 2,607 have locally available photos. The ResNet18 features are pretrained on ImageNet rather than Zimbabwean property photos, and the text embeddings are learned from this small training split rather than a pretrained language model. The listings are mostly from Harare and are asking prices, not completed transactions. Validate on repeated or time-based splits and verified sale prices before using these estimates operationally.